In [ ]:
import plotly.graph_objects as go
import numpy as np

def create_cube(x, y, z, size=1, color='rgba(0, 0, 255, 0.3)', block_id='cube'):
    """Returns a Mesh3d object for a transparent cube at the specified location."""
    # Cube corners
    vertices = np.array([
        [x, y, z],  # 0
        [x + size, y, z],  # 1
        [x + size, y + size, z],  # 2
        [x, y + size, z],  # 3
        [x, y, z + size],  # 4
        [x + size, y, z + size],  # 5
        [x + size, y + size, z + size],  # 6
        [x, y + size, z + size],  # 7
    ])

    # Define the 12 triangles composing the 6 cube faces
    I = [0, 0, 0, 1, 1, 2, 3, 4, 4, 5, 6, 7]
    J = [1, 3, 4, 2, 5, 3, 2, 5, 7, 6, 7, 6]
    K = [3, 4, 7, 3, 6, 2, 7, 6, 6, 7, 3, 2]

    x_coords = vertices[:, 0]
    y_coords = vertices[:, 1]
    z_coords = vertices[:, 2]

    return go.Mesh3d(
        x=x_coords, y=y_coords, z=z_coords,
        i=I, j=J, k=K,
        opacity=0.3,
        color=color,
        name=block_id,
        hovertext=block_id,
        hoverinfo='text',
        flatshading=True,
        showscale=False
    )

# Generate 3x3x3 rubik-style cube grid
shapes = []
for i in range(3):
    for j in range(3):
        for k in range(3):
            block_id = f"block_{i}_{j}_{k}"
            cube = create_cube(i, j, k, color='rgba(0,0,255,0.3)', block_id=block_id)
            shapes.append(cube)

# Plot
fig = go.Figure(data=shapes)
fig.update_layout(
    scene=dict(
        xaxis=dict(title='X'),
        yaxis=dict(title='Y'),
        zaxis=dict(title='Z'),
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=30),
    title="Transparent 3x3x3 Rubik’s Cube Blocks"
)
fig.show()


In [ ]:
import plotly.graph_objects as go
import numpy as np

def create_cube_with_wireframe(x, y, z, size=1, color='rgba(0, 0, 255, 0.3)', block_id='cube'):
    """Returns Mesh3d and wireframe line traces for a transparent cube."""
    # Define 8 cube corners
    vertices = np.array([
        [x, y, z],  # 0
        [x + size, y, z],  # 1
        [x + size, y + size, z],  # 2
        [x, y + size, z],  # 3
        [x, y, z + size],  # 4
        [x + size, y, z + size],  # 5
        [x + size, y + size, z + size],  # 6
        [x, y + size, z + size],  # 7
    ])

    # Define triangles for cube faces
    I = [0, 0, 0, 1, 1, 2, 3, 4, 4, 5, 6, 7]
    J = [1, 3, 4, 2, 5, 3, 2, 5, 7, 6, 7, 6]
    K = [3, 4, 7, 3, 6, 2, 7, 6, 6, 7, 3, 2]

    x_coords = vertices[:, 0]
    y_coords = vertices[:, 1]
    z_coords = vertices[:, 2]

    mesh = go.Mesh3d(
        x=x_coords, y=y_coords, z=z_coords,
        i=I, j=J, k=K,
        opacity=0.3,
        color=color,
        name=block_id,
        hovertext=block_id,
        hoverinfo='text',
        flatshading=True,
        showscale=False
    )

    # Define 12 edges for the wireframe
    edges = [
        (0,1), (1,2), (2,3), (3,0),  # Bottom face
        (4,5), (5,6), (6,7), (7,4),  # Top face
        (0,4), (1,5), (2,6), (3,7)   # Vertical edges
    ]

    # Create Scatter3d lines for edges
    wireframe_lines = []
    for start, end in edges:
        line = go.Scatter3d(
            x=[vertices[start][0], vertices[end][0]],
            y=[vertices[start][1], vertices[end][1]],
            z=[vertices[start][2], vertices[end][2]],
            mode='lines',
            line=dict(color='black', width=2),
            showlegend=False,
            hoverinfo='skip'
        )
        wireframe_lines.append(line)

    return [mesh] + wireframe_lines

# Generate the full 3x3x3 cube with outlines
shapes = []
for i in range(3):
    for j in range(3):
        for k in range(3):
            block_id = f"block_{i}_{j}_{k}"
            components = create_cube_with_wireframe(i, j, k, block_id=block_id)
            shapes.extend(components)

# Plot the result
fig = go.Figure(data=shapes)
fig.update_layout(
    scene=dict(
        xaxis=dict(title='X'),
        yaxis=dict(title='Y'),
        zaxis=dict(title='Z'),
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=30),
    title="Rubik’s Cube Style 3D Blocks with Wireframes"
)
fig.show()


In [ ]:
import plotly.graph_objects as go
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# --- Parameters ---
block_size = (5, 5, 3)
num_blocks = (4, 3, 2)
dx, dy, dz = block_size
nx, ny, nz = num_blocks

# --- Generate all cube traces ---
traces = []

def create_cube_with_wireframe(x, y, z, dx, dy, dz, fill_opacity=0.3, line_opacity=1.0):
    vertices = np.array([
        [x, y, z], [x + dx, y, z], [x + dx, y + dy, z], [x, y + dy, z],
        [x, y, z + dz], [x + dx, y, z + dz], [x + dx, y + dy, z + dz], [x, y + dy, z + dz],
    ])

    I = [0, 0, 0, 1, 1, 2, 3, 4, 4, 5, 6, 7]
    J = [1, 3, 4, 2, 5, 3, 2, 5, 7, 6, 7, 6]
    K = [3, 4, 7, 3, 6, 2, 7, 6, 6, 7, 3, 2]

    mesh = go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=I, j=J, k=K,
        opacity=fill_opacity,
        color='blue',
        flatshading=True,
        hoverinfo='skip',
        showscale=False
    )

    # Wireframe as lines
    edges = [
        (0,1), (1,2), (2,3), (3,0),
        (4,5), (5,6), (6,7), (7,4),
        (0,4), (1,5), (2,6), (3,7)
    ]
    lines = []
    for start, end in edges:
        line = go.Scatter3d(
            x=[vertices[start][0], vertices[end][0]],
            y=[vertices[start][1], vertices[end][1]],
            z=[vertices[start][2], vertices[end][2]],
            mode='lines',
            line=dict(color='black', width=2),
            opacity=line_opacity,
            hoverinfo='skip',
            showlegend=False
        )
        lines.append(line)

    return [mesh] + lines

# Create all blocks
for i in range(nx):
    for j in range(ny):
        for k in range(nz):
            x = i * dx
            y = j * dy
            z = k * dz
            traces.extend(create_cube_with_wireframe(x, y, z, dx, dy, dz))

# --- Set up FigureWidget ---
fig = go.FigureWidget(data=traces)
fig.update_layout(
    scene=dict(
        xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z'),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=30),
    title="Interactive Mining Block Model with Transparency Sliders"
)

# --- Sliders ---
fill_slider = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description='Block Fill')
wire_slider = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description='Wireframe')

# --- Live update function ---
def update_opacity(*args):
    for i, trace in enumerate(fig.data):
        if isinstance(trace, go.Mesh3d):
            fig.data[i].opacity = fill_slider.value
        elif isinstance(trace, go.Scatter3d):
            fig.data[i].opacity = wire_slider.value

fill_slider.observe(update_opacity, names='value')
wire_slider.observe(update_opacity, names='value')

# --- Display everything ---
display(fig, fill_slider, wire_slider)
update_opacity()


FigureWidget({
    'data': [{'color': 'blue',
              'flatshading': True,
              'hoverinfo': 'skip',
              'i': [0, 0, 0, 1, 1, 2, 3, 4, 4, 5, 6, 7],
              'j': [1, 3, 4, 2, 5, 3, 2, 5, 7, 6, 7, 6],
              'k': [3, 4, 7, 3, 6, 2, 7, 6, 6, 7, 3, 2],
              'opacity': 0.3,
              'showscale': False,
              'type': 'mesh3d',
              'uid': '89691405-fc95-40eb-9a2a-3ef3321723c1',
              'x': array([0, 5, 5, 0, 0, 5, 5, 0]),
              'y': array([0, 0, 5, 5, 0, 0, 5, 5]),
              'z': array([0, 0, 0, 0, 3, 3, 3, 3])},
             {'hoverinfo': 'skip',
              'line': {'color': 'black', 'width': 2},
              'mode': 'lines',
              'opacity': 1.0,
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'c3363ddf-0961-47a0-8858-fd741978d420',
              'x': [0, 5],
              'y': [0, 0],
              'z': [0, 0]},
             {'hoverinfo': 'skip',
 

FloatSlider(value=0.3, description='Block Fill', max=1.0, step=0.05)

FloatSlider(value=1.0, description='Wireframe', max=1.0, step=0.05)